# Fitting a multi-state income-protection model with `jact.fitting`

This notebook demonstrates the complete modelling-framework boundary on a deliberately rich synthetic insurance example. We generate monthly interval histories across active, disabled, critical-illness, lapsed, and dead states. Eleven transition hazards contain nonlinear attained-age, BMI, income, calendar-time, and duration relationships plus demographic, occupational, geographic, and cross-feature interactions. We then split by subject, fit all transition log-hazards jointly, export a portable artifact, and use the reconstructed model in JACT's probability solver and simulator.

The data are synthetic so the notebook is reproducible; the workflow is the same for canonicalized production event histories.

## 1. Setup and topology

The state space includes recovery cycles between active, disabled, and critical illness. Lapse and death are distinct absorbing states. The fitting topology is derived from the exact `StateSpace` that will later be used for valuation.

In [ ]:
from pathlib import Path

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

import jact

print("JACT", jact.__version__, "| JAX", jax.__version__, "| devices", jax.devices())

In [ ]:
state_space = jact.StateSpace(
    states=["active", "disabled", "critical", "lapsed", "dead"],
    transitions=[
        ("active", "disabled"),
        ("active", "critical"),
        ("active", "lapsed"),
        ("active", "dead"),
        ("disabled", "active"),
        ("disabled", "critical"),
        ("disabled", "lapsed"),
        ("disabled", "dead"),
        ("critical", "disabled"),
        ("critical", "lapsed"),
        ("critical", "dead"),
    ],
)
topology = jact.fitting.TopologySpec.from_state_space(state_space)
edge_id = {edge: i for i, edge in enumerate(topology.edges)}

print("Topology fingerprint:", topology.fingerprint[:16] + "…")
for i, edge in enumerate(topology.edges):
    print(f"edge {i}: {edge[0]} -> {edge[1]}")

## 2. Generate canonical interval histories

Each subject is observed monthly for up to six years. Some enter while already disabled or critically ill with non-zero current-state duration. Hazards contain nonlinear age, BMI, income, calendar-time, and duration effects, plus smoking, sex, occupation, region, and interaction effects. Events occur under the exact competing-risks probability for the month's midpoint-constant hazards. Covariates on an event row are the pre-transition values. Several edge types are deliberately rare, making shared representation and partial pooling practically useful.

In [ ]:
rng = np.random.default_rng(20260825)
n_subjects = 6_000
months = 6 * 12
dt = 1.0 / 12.0

entry_age = np.clip(rng.normal(48.0, 11.0, n_subjects), 23.0, 74.0)
sex_label = rng.choice(["female", "male"], n_subjects, p=[0.48, 0.52])
smoker_label = rng.choice(["never", "former", "current"], n_subjects, p=[0.60, 0.22, 0.18])
occupation_label = rng.choice(["office", "manual", "hazardous"], n_subjects, p=[0.55, 0.34, 0.11])
region_label = rng.choice(["north", "south", "east", "west"], n_subjects)
bmi = np.clip(25.0 + rng.normal(0.0, 4.2, n_subjects) + 1.1 * (smoker_label == "current"), 17.0, 42.0)
income_k = np.clip(
    rng.lognormal(np.log(58.0), 0.42, n_subjects)
    * np.where(occupation_label == "office", 1.18, 0.88),
    18.0, 220.0,
)

active = topology.state_id("active")
disabled = topology.state_id("disabled")
critical = topology.state_id("critical")
lapsed = topology.state_id("lapsed")
dead = topology.state_id("dead")

def true_exits(state, clock, attained_age, duration, sex, smoking, occupation, region, bmi_value, income_value):
    age = (attained_age - 50.0) / 10.0
    bmi_risk = (bmi_value - 24.0) / 5.0
    income = np.log(income_value / 55.0)
    male = sex == "male"
    former = smoking == "former"
    current = smoking == "current"
    manual = occupation == "manual"
    hazardous = occupation == "hazardous"
    economic_cycle = np.sin(2.0 * np.pi * clock / 4.5)
    claims_wave = np.exp(-0.5 * ((clock - 3.2) / 0.7) ** 2)
    if state == active:
        edges = [
            edge_id[("active", "disabled")],
            edge_id[("active", "critical")],
            edge_id[("active", "lapsed")],
            edge_id[("active", "dead")],
        ]
        rates = np.array([
            0.045 * np.exp(0.20 * age + 0.10 * age**2 + 0.15 * bmi_risk**2 + 0.35 * manual + 0.75 * hazardous + 0.45 * current + 0.22 * male * current + 0.18 * claims_wave + 0.15 * hazardous * bmi_risk),
            0.006 * np.exp(0.45 * age + 0.16 * age**2 + 0.30 * current + 0.18 * male + 0.14 * bmi_risk**2),
            0.055 * np.exp(-0.22 * age - 0.20 * income + 0.30 * economic_cycle + 0.18 * (region == "west") + 0.12 * current),
            0.006 * np.exp(0.72 * age + 0.16 * age**2 + 0.55 * current + 0.20 * former + 0.18 * male + 0.12 * bmi_risk**2),
        ])
    elif state == disabled:
        edges = [
            edge_id[("disabled", "active")],
            edge_id[("disabled", "critical")],
            edge_id[("disabled", "lapsed")],
            edge_id[("disabled", "dead")],
        ]
        rates = np.array([
            0.42 * np.exp(-0.35 * age - 0.40 * manual - 0.75 * hazardous - 0.35 * current - 0.28 * duration + 0.45 * np.exp(-((duration - 0.25) / 0.25) ** 2) + 0.12 * income),
            0.025 * np.exp(0.42 * age + 0.25 * current + 0.18 * bmi_risk**2 + 0.10 * duration),
            0.035 * np.exp(-0.15 * age - 0.10 * income + 0.20 * economic_cycle + 0.15 * (region == "west") + 0.10 * duration),
            0.035 * np.exp(0.65 * age + 0.45 * current + 0.16 * bmi_risk**2 + 0.12 * duration + 0.18 * hazardous),
        ])
    else:
        edges = [
            edge_id[("critical", "disabled")],
            edge_id[("critical", "lapsed")],
            edge_id[("critical", "dead")],
        ]
        rates = np.array([
            0.22 * np.exp(-0.30 * age - 0.20 * manual - 0.40 * hazardous - 0.25 * current - 0.15 * duration + 0.25 * (sex == "female")),
            0.025 * np.exp(-0.10 * age - 0.15 * income + 0.16 * economic_cycle + 0.12 * duration),
            0.14 * np.exp(0.60 * age + 0.50 * current + 0.16 * bmi_risk**2 + 0.50 * np.exp(-2.0 * duration) + 0.08 * duration),
        ])
    return edges, rates

rows = {name: [] for name in (
    "subject_id", "source_state", "event_edge",
    "t0", "t1", "d0", "d1", "entry_age", "bmi", "income_k",
    "sex", "smoker", "occupation", "region"
)}

for subject in range(n_subjects):
    initial_draw = rng.random()
    state = active if initial_draw < 0.90 else (disabled if initial_draw < 0.98 else critical)
    duration = 0.0 if state == active else rng.uniform(0.0, 1.5 if state == disabled else 0.5)
    for month in range(months):
        if state in (lapsed, dead):
            break
        t0 = month * dt
        t1 = t0 + dt
        d0 = duration
        d1 = duration + dt
        attained_age = entry_age[subject] + 0.5 * (t0 + t1)
        exits, rates = true_exits(
            state, 0.5 * (t0 + t1), attained_age, 0.5 * (d0 + d1),
            sex_label[subject], smoker_label[subject], occupation_label[subject],
            region_label[subject], bmi[subject], income_k[subject],
        )
        total_rate = rates.sum()
        event = -1
        if rng.random() < 1.0 - np.exp(-total_rate * dt):
            event = exits[rng.choice(len(exits), p=rates / total_rate)]

        values = (
            subject, state, event, t0, t1, d0, d1, entry_age[subject], bmi[subject],
            income_k[subject], sex_label[subject], smoker_label[subject],
            occupation_label[subject], region_label[subject],
        )
        for name, value in zip(rows, values):
            rows[name].append(value)

        if event >= 0:
            state = topology.edge_targets[event]
            duration = 0.0
        else:
            duration = d1

rows = {name: np.asarray(values) for name, values in rows.items()}
event_counts = np.bincount(
    rows["event_edge"][rows["event_edge"] >= 0], minlength=topology.n_edges
)

print(f"{len(rows['t0']):,} at-risk intervals from {n_subjects:,} subjects")
for edge, count in zip(topology.edges, event_counts):
    print(f"{edge[0]:>8} -> {edge[1]:<8}: {count:4d} events")

## 3. Split subjects and encode predictors

The split is made at subject level, never at interval level. The encoder is fitted only on training subjects and owns the numeric scaling and categorical vocabulary used by both training and exported inference.

In [ ]:
split_rng = np.random.default_rng(8675309)
subject_order = split_rng.permutation(n_subjects)
n_train_subjects = int(0.8 * n_subjects)
train_subjects = subject_order[:n_train_subjects]
validation_subjects = subject_order[n_train_subjects:]
train_mask = np.isin(rows["subject_id"], train_subjects)
validation_mask = np.isin(rows["subject_id"], validation_subjects)

encoder = jact.fitting.FeatureEncoder.fit(
    numeric={
        "entry_age": entry_age[train_subjects],
        "bmi": bmi[train_subjects],
        "income_k": income_k[train_subjects],
    },
    categorical={
        "sex": sex_label[train_subjects],
        "smoker": smoker_label[train_subjects],
        "occupation": occupation_label[train_subjects],
        "region": region_label[train_subjects],
    },
)
covariates = encoder.transform_columns({
    "entry_age": rows["entry_age"],
    "bmi": rows["bmi"],
    "income_k": rows["income_k"],
    "sex": rows["sex"],
    "smoker": rows["smoker"],
    "occupation": rows["occupation"],
    "region": rows["region"],
})

print("Encoded columns:", encoder.feature_names)
print("Training rows:", train_mask.sum(), "| validation rows:", validation_mask.sum())

In [ ]:
def canonical_batch(indices, *, valid=None):
    indices = np.asarray(indices)
    if valid is None:
        valid = np.ones(len(indices), dtype=bool)
    return jact.fitting.CanonicalBatch.from_arrays(
        subject_id=rows["subject_id"][indices],
        source_state=rows["source_state"][indices],
        event_edge=rows["event_edge"][indices],
        t0=rows["t0"][indices],
        t1=rows["t1"][indices],
        d0=rows["d0"][indices],
        d1=rows["d1"][indices],
        covariates=covariates[indices],
        valid=valid,
        dtype=jnp.float32,
    ).validate(topology)

def fixed_batches(mask, batch_size, seed):
    indices = np.flatnonzero(mask)
    np.random.default_rng(seed).shuffle(indices)
    batches = []
    for start in range(0, len(indices), batch_size):
        chunk = indices[start : start + batch_size]
        valid = np.arange(batch_size) < len(chunk)
        padded = np.full(batch_size, chunk[0], dtype=int)
        padded[: len(chunk)] = chunk
        batches.append(canonical_batch(padded, valid=valid))
    return tuple(batches)

batch_size = 8_192
training_batches = fixed_batches(train_mask, batch_size, seed=1)
validation_batches = fixed_batches(validation_mask, batch_size, seed=2)
full_training_batch = canonical_batch(np.flatnonzero(train_mask))

print(len(training_batches), "fixed training batches of shape", training_batches[0].t0.shape)
print(len(validation_batches), "fixed validation batches")

## 4. Joint structured log-hazard model

Every forward pass returns all eleven edge log-hazards. The likelihood masks them by source state, sums only valid competing exits, and gathers the observed event edge in log space. Separate time and duration splines are combined with explicit encoded effects and a shared SiLU residual that can learn nonlinearities and interactions through low-rank edge heads.

In [ ]:
config = jact.fitting.StructuredModelConfig(
    n_features=encoder.n_features,
    time_knots=tuple(float(x) for x in np.linspace(0.0, 6.0, 9)),
    duration_knots=tuple(float(x) for x in np.linspace(0.0, 6.0, 9)),
    embedding_size=6,
    hidden_size=32,
    rank=8,
    time_center=3.0,
    time_scale=3.0,
    duration_center=3.0,
    duration_scale=3.0,
)
log_hazard_model = jact.fitting.StructuredLogHazardModel(topology, config)
initial_intercepts = jact.fitting.empirical_intercepts(
    full_training_batch, topology
)
initial_params = log_hazard_model.init(
    jax.random.key(31415), intercept=initial_intercepts
)

for edge, value in zip(topology.edges, np.exp(np.asarray(initial_intercepts))):
    print(f"initial {edge[0]:>8} -> {edge[1]:<8}: {value:.4f} per year")

## 5. Fit with midpoint competing-risks likelihood

The monthly rows are already fine and split at every duration update, so midpoint piecewise-exponential fitting is appropriate. Smoothness and weight penalties are explicit parts of the objective. The trainer uses gradient clipping, AdamW, warmup, cosine decay, EMA parameters, and validation early stopping.

In [ ]:
def objective(params, batch):
    likelihood = jact.fitting.piecewise_exponential_nll(
        log_hazard_model, params, batch, topology
    ).loss
    regularization = log_hazard_model.penalty(
        params, smoothness=1e-3, weight_decay=1e-5, edge_specific=1e-4
    )
    return likelihood + regularization

epochs = 10
optimizer = jact.fitting.AdamWConfig(
    learning_rate=2e-3,
    weight_decay=1e-4,
    clip_norm=5.0,
    warmup_steps=10,
    total_steps=epochs * len(training_batches),
    minimum_learning_rate_ratio=0.1,
    ema_decay=0.98,
)
trainer = jact.fitting.Trainer(objective, optimizer)
fit = trainer.fit(
    initial_params,
    training_batches,
    epochs=epochs,
    validation_batches=validation_batches,
    patience=4,
)

print("Best epoch:", fit.best_epoch + 1)
print("Final validation objective:", fit.history.validation_loss[-1])

In [ ]:
figure, axis = plt.subplots(figsize=(7, 4))
epoch_axis = np.arange(1, len(fit.history.training_loss) + 1)
axis.plot(epoch_axis, fit.history.training_loss, marker="o", label="training")
axis.plot(epoch_axis, fit.history.validation_loss, marker="o", label="validation")
axis.set(xlabel="epoch", ylabel="penalized negative log-likelihood", title="Fitting history")
axis.grid(alpha=0.25)
axis.legend()
plt.show()

A fixed quadrature likelihood is useful when hazards move materially inside a row. On monthly rows, its result should be close to midpoint fitting; this is a diagnostic, not a replacement for checking JACT's deployment grid.

In [ ]:
diagnostic_batch = validation_batches[0]
midpoint_nll = jact.fitting.piecewise_exponential_nll(
    log_hazard_model, fit.params, diagnostic_batch, topology
).loss
quadrature_nll = jact.fitting.fixed_quadrature_nll(
    log_hazard_model, fit.params, diagnostic_batch, topology, order=4
).loss
print("midpoint NLL: ", float(midpoint_nll))
print("quadrature NLL:", float(quadrature_nll))
print("absolute difference:", float(jnp.abs(midpoint_nll - quadrature_nll)))

## 6. Duration-axis occurrence/exposure calibration

For each source state, duration cut points are chosen so its validation exposure is spread approximately equally across six bins. Points are occurrence/exposure rates and lines are exposure-weighted mean fitted intensities for exactly the same at-risk rows. Point area represents bin exposure and vertical bars are approximate 95% Poisson intervals. This is a case-mix-matched comparison: clock time and all seven portfolio predictors vary as observed rather than being fixed to one profile. Empirical bins are suppressed when an edge has fewer than 15 validation events; a granular calibration curve would be misleading in that case.

In [ ]:
validation_batch = canonical_batch(np.flatnonzero(validation_mask))
validation_log_hazards = jax.jit(log_hazard_model)(
    fit.params,
    validation_batch.source_state,
    validation_batch.midpoint_t,
    validation_batch.midpoint_d,
    validation_batch.covariates,
)
validation_intensities = np.exp(
    np.clip(np.asarray(validation_log_hazards), -20.0, 5.0)
)
validation_source = np.asarray(validation_batch.source_state)
validation_event = np.asarray(validation_batch.event_edge)
validation_duration = np.asarray(validation_batch.midpoint_d)
validation_exposure = np.asarray(validation_batch.exposure)

def exposure_quantile_breaks(duration, exposure, bins=6):
    order = np.argsort(duration)
    ordered_duration = duration[order]
    ordered_exposure = exposure[order]
    cumulative = (np.cumsum(ordered_exposure) - 0.5 * ordered_exposure) / ordered_exposure.sum()
    breaks = np.interp(np.linspace(0.0, 1.0, bins + 1), cumulative, ordered_duration)
    breaks[0] = ordered_duration[0] - 1e-6
    breaks[-1] = ordered_duration[-1] + 1e-6
    return np.unique(breaks)

for source in state_space.transient:
    source_id = topology.state_id(source)
    source_rows = validation_source == source_id
    source_edges = topology.outgoing_edges[source_id]
    duration_breaks = exposure_quantile_breaks(
        validation_duration[source_rows], validation_exposure[source_rows]
    )
    figure, axes = plt.subplots(
        1, len(source_edges), figsize=(4.2 * len(source_edges), 3.8),
        sharex=True, squeeze=False,
    )
    axes = axes[0]

    for edge_index, axis in zip(source_edges, axes):
        target = topology.edges[edge_index][1]
        n_bins = len(duration_breaks) - 1
        duration_centers = np.zeros(n_bins)
        occurrence_rate = np.zeros(n_bins)
        fitted_rate = np.zeros(n_bins)
        exposure_by_bin = np.zeros(n_bins)
        occurrence_by_bin = np.zeros(n_bins)

        for duration_bin, (left, right) in enumerate(zip(duration_breaks[:-1], duration_breaks[1:])):
            at_risk = (
                source_rows
                & (validation_duration >= left)
                & (validation_duration < right)
            )
            exposure = validation_exposure[at_risk].sum()
            occurrences = np.sum(at_risk & (validation_event == edge_index))
            exposure_by_bin[duration_bin] = exposure
            occurrence_by_bin[duration_bin] = occurrences
            duration_centers[duration_bin] = np.average(
                validation_duration[at_risk], weights=validation_exposure[at_risk]
            )
            occurrence_rate[duration_bin] = occurrences / exposure
            fitted_rate[duration_bin] = np.average(
                validation_intensities[at_risk, edge_index],
                weights=validation_exposure[at_risk],
            )

        axis.plot(
            duration_centers, fitted_rate, linewidth=2.2,
            label="mean fitted intensity",
        )
        total_occurrences = int(occurrence_by_bin.sum())
        if total_occurrences >= 15:
            lower_rate = (
                np.maximum(np.sqrt(occurrence_by_bin) - 0.98, 0.0) ** 2
                / exposure_by_bin
            )
            upper_rate = (
                (np.sqrt(occurrence_by_bin + 1.0) + 0.98) ** 2
                / exposure_by_bin
            )
            axis.vlines(
                duration_centers, lower_rate, upper_rate,
                color="tab:orange", alpha=0.55, linewidth=1.2,
            )
            point_size = 30.0 + 80.0 * exposure_by_bin / exposure_by_bin.max()
            axis.scatter(
                duration_centers, occurrence_rate, s=point_size,
                color="tab:orange", edgecolor="white", linewidth=0.7,
                label="occurrences / exposure", zorder=3,
            )
        else:
            axis.text(
                0.5, 0.92,
                f"{total_occurrences} validation events\nempirical bins suppressed",
                transform=axis.transAxes, ha="center", va="top",
                fontsize=9, color="tab:orange",
            )

        axis.set_title(f"{source} → {target}")
        axis.set_xlabel("current-state duration (years)")
        axis.set_ylim(bottom=0.0)
        axis.grid(alpha=0.25)

    axes[0].set_ylabel("annual rate / intensity")
    axes[0].legend(fontsize=8, loc="best")
    figure.suptitle(f"Out-of-sample duration calibration: {source} exits")
    figure.tight_layout()
    plt.show()

## 7. Export, reload, and validate the artifact

The artifact stores arrays and declarative metadata in a versioned NPZ file without pickle. A topology fingerprint prevents accidental reconstruction against reordered states or edges. In a production pipeline, fitting history and optimizer state would be checkpointed separately.

In [ ]:
artifact = jact.fitting.FittedArtifact(
    topology=topology,
    encoder=encoder,
    model_config=config,
    params=fit.params,
    time_unit="years",
    link="exp",
    log_hazard_bounds=(-20.0, 5.0),
    feature_dtypes={
        "entry_age": "float32", "bmi": "float32", "income_k": "float32",
        "sex": "int32", "smoker": "int32", "occupation": "int32", "region": "int32",
    },
    feature_units={
        "entry_age": "years", "bmi": "kg/m²", "income_k": "thousands/year",
        "sex": "category code", "smoker": "category code",
        "occupation": "category code", "region": "category code",
    },
    supported_ranges={
        "entry_age": (23.0, 74.0), "bmi": (17.0, 42.0),
        "income_k": (18.0, 220.0), "t": (0.0, 10.0), "d": (0.0, 10.0),
    },
    fitting_metadata={
        "example": "synthetic multi-state income protection",
        "seed": 20260825,
        "event_counts": {
            f"{source}->{target}": int(count)
            for (source, target), count in zip(topology.edges, event_counts)
        },
    },
).validate()

artifact_path = Path("synthetic_income_protection_model.jact.npz")
artifact.save(artifact_path)
loaded = jact.fitting.FittedArtifact.load(artifact_path)
print("Saved", artifact_path, "with topology", loaded.topology.fingerprint[:16] + "…")

In [ ]:
sex_female, sex_male = map(int, encoder.encode_category("sex", ["female", "male"]))
smoker_never, smoker_former, smoker_current = map(
    int, encoder.encode_category("smoker", ["never", "former", "current"])
)
occupation_office, occupation_manual, occupation_hazardous = map(
    int, encoder.encode_category("occupation", ["office", "manual", "hazardous"])
)
region_north, region_south, region_east, region_west = map(
    int, encoder.encode_category("region", ["north", "south", "east", "west"])
)
duration_grid = jnp.linspace(0.0, 10.0, 121)[None, :]

loaded.adapter().validate_hazards(
    state_space,
    t=jnp.asarray(10.0),
    d=duration_grid,
    covariates={
        "entry_age": jnp.array([23.0, 50.0, 74.0]),
        "bmi": jnp.array([17.0, 27.0, 42.0]),
        "income_k": jnp.array([18.0, 60.0, 220.0]),
        "sex": jnp.array([sex_female, sex_male, sex_male]),
        "smoker": jnp.array([smoker_never, smoker_current, smoker_current]),
        "occupation": jnp.array([occupation_office, occupation_hazardous, occupation_hazardous]),
        "region": jnp.array([region_north, region_west, region_west]),
    },
)
print("Raw and wrapped hazards are finite on the stress grid.")

active_exits = loaded.adapter().exit_intensity(state_space, "active")
hazards = active_exits(
    jnp.asarray(2.0), duration_grid, entry_age=jnp.asarray(50.0), bmi=jnp.asarray(27.0),
    income_k=jnp.asarray(60.0), sex=jnp.asarray(sex_female), smoker=jnp.asarray(smoker_never),
    occupation=jnp.asarray(occupation_office), region=jnp.asarray(region_north),
)[:, 0, :]
figure, axis = plt.subplots(figsize=(7, 4))
for target, values in zip(state_space.targets("active"), np.asarray(hazards)):
    axis.plot(np.asarray(duration_grid[0]), values, label=f"active → {target}")
axis.set(xlabel="duration active (years)", ylabel="annual intensity", title="Exported active-state hazards")
axis.grid(alpha=0.25)
axis.legend()
plt.show()

## 8. Solve portfolio probabilities

The reconstructed callables combine seven portfolio predictors with solver clock time, source state, and reset current-state duration on JACT's `(batch, duration)` grid. We compare four deliberately different policyholder profiles over ten years.

In [ ]:
model = loaded.build_model(state_space, assignment="exits")
profile_names = (
    "35F, office, never smoker", "50M, office, former smoker",
    "50F, manual, current smoker", "65M, hazardous, current smoker",
)
portfolio_age = jnp.array([35.0, 50.0, 50.0, 65.0])
portfolio_bmi = jnp.array([22.0, 27.0, 31.0, 35.0])
portfolio_income = jnp.array([85.0, 65.0, 42.0, 35.0])
portfolio_sex = jnp.array([sex_female, sex_male, sex_female, sex_male])
portfolio_smoker = jnp.array([smoker_never, smoker_former, smoker_current, smoker_current])
portfolio_occupation = jnp.array([occupation_office, occupation_office, occupation_manual, occupation_hazardous])
portfolio_region = jnp.array([region_north, region_east, region_south, region_west])

result = model.solve(
    initial="active",
    horizon=10,
    steps_per_unit=12,
    entry_age=portfolio_age,
    bmi=portfolio_bmi,
    income_k=portfolio_income,
    sex=portfolio_sex,
    smoker=portfolio_smoker,
    occupation=portfolio_occupation,
    region=portfolio_region,
)
times = np.linspace(0.0, 10.0, result.probability.shape[0])

figure, axes = plt.subplots(2, 2, figsize=(11, 7), sharex=True, sharey=True)
for profile, axis in enumerate(axes.flat):
    for state_index, state in enumerate(result.states):
        axis.plot(times, np.asarray(result.probability[:, profile, state_index]), label=state)
    axis.set_title(profile_names[profile])
    axis.grid(alpha=0.25)
    axis.set_ylim(0.0, 1.0)
axes[1, 0].set_xlabel("years")
axes[1, 1].set_xlabel("years")
axes[0, 0].set_ylabel("probability")
axes[1, 0].set_ylabel("probability")
axes[0, 0].legend()
figure.suptitle("Fitted multi-state probabilities")
figure.tight_layout()
plt.show()

## 9. Deployment-grid and simulation checks

Fitting integration accuracy and JACT midpoint resolution are separate questions. First compare terminal probabilities over increasingly fine deployment grids. Then make a like-for-like six-year comparison of actual validation outcomes, fitted probabilities, and simulated outcomes using each validation subject's observed initial state, starting duration, and covariates.

In [ ]:
terminal_by_resolution = {}
for steps_per_unit in (6, 12, 24):
    convergence_result = model.solve(
        initial="active", horizon=10, steps_per_unit=steps_per_unit,
        entry_age=jnp.array([50.0]), bmi=jnp.array([27.0]), income_k=jnp.array([65.0]),
        sex=jnp.array([sex_male]), smoker=jnp.array([smoker_former]),
        occupation=jnp.array([occupation_office]), region=jnp.array([region_east]),
    )
    terminal_by_resolution[steps_per_unit] = np.asarray(convergence_result.probability[-1, 0])

for resolution, values in terminal_by_resolution.items():
    print(f"{resolution:2d} steps/year:", dict(zip(result.states, values.round(6))))
print("max |12 - 24|:", np.max(np.abs(terminal_by_resolution[12] - terminal_by_resolution[24])))

In [ ]:
validation_subject_ids = np.sort(validation_subjects)
first_row = np.searchsorted(rows["subject_id"], validation_subject_ids, side="left")
last_row = np.searchsorted(rows["subject_id"], validation_subject_ids, side="right") - 1
initial_state = rows["source_state"][first_row]
initial_duration = rows["d0"][first_row]
last_source = rows["source_state"][last_row]
last_event = rows["event_edge"][last_row]
actual_terminal_state = last_source.copy()
ended_with_event = last_event >= 0
actual_terminal_state[ended_with_event] = np.asarray(topology.edge_targets)[
    last_event[ended_with_event]
]
actual_proportion = np.bincount(
    actual_terminal_state, minlength=topology.n_states
) / len(validation_subject_ids)

validation_initial = state_space.initial_per_individual(
    state_indices=jnp.asarray(initial_state, dtype=jnp.int32),
    duration=jnp.asarray(initial_duration, dtype=jnp.float32),
    initial_states=("active", "disabled", "critical"),
)
validation_covariates = {
    "entry_age": jnp.asarray(entry_age[validation_subject_ids]),
    "bmi": jnp.asarray(bmi[validation_subject_ids]),
    "income_k": jnp.asarray(income_k[validation_subject_ids]),
    "sex": encoder.encode_category("sex", sex_label[validation_subject_ids]),
    "smoker": encoder.encode_category("smoker", smoker_label[validation_subject_ids]),
    "occupation": encoder.encode_category("occupation", occupation_label[validation_subject_ids]),
    "region": encoder.encode_category("region", region_label[validation_subject_ids]),
}

observed_horizon = 6
validation_solved = model.solve(
    initial=validation_initial, horizon=observed_horizon, steps_per_unit=12,
    **validation_covariates,
)
solved_proportion = np.asarray(validation_solved.probability[-1]).mean(axis=0)

simulated = model.simulate(
    initial=validation_initial,
    horizon=observed_horizon,
    steps_per_unit=12,
    max_jumps=16,
    replicates=50,
    key=jax.random.key(2026),
    **validation_covariates,
)
simulated_proportion = np.bincount(
    np.asarray(simulated.final_state).ravel(), minlength=topology.n_states
) / simulated.final_state.size

print("state       actual     solved   simulated")
for state, actual_value, solved_value, simulated_value in zip(
    validation_solved.states, actual_proportion, solved_proportion, simulated_proportion
):
    print(f"{state:<9} {actual_value:8.4f}   {solved_value:8.4f}   {simulated_value:8.4f}")
print("overflow fraction:", float(jnp.mean(simulated.overflow)))

state_axis = np.arange(topology.n_states)
width = 0.26
figure, axis = plt.subplots(figsize=(9, 4))
axis.bar(state_axis - width, actual_proportion, width, label="actual validation data")
axis.bar(state_axis, solved_proportion, width, label="mean fitted probability")
axis.bar(state_axis + width, simulated_proportion, width, label="simulation")
axis.set_xticks(state_axis, validation_solved.states)
axis.set(ylabel="proportion at year 6", title="Observed, solved, and simulated terminal states")
axis.set_ylim(0.0, 1.0)
axis.grid(axis="y", alpha=0.25)
axis.legend()
plt.show()

## What this example established

- State and edge identifiers came from one deterministic JACT topology.
- Rows preserved clock time and current-state duration separately, including duration resets.
- The data included eleven edge types, two absorbing outcomes, recovery cycles, and prevalent-state entries with non-zero duration.
- Nonlinear continuous effects and interactions were learned alongside demographic, occupational, and geographic categories.
- Subject-level splitting prevented interval leakage.
- The objective was a continuous-time competing-risks likelihood, not independent edge classifiers.
- Out-of-sample occurrence/exposure rates were compared with case-mix-matched fitted intensities over current-state duration.
- Preprocessing, units, architecture, parameters, ranges, and topology metadata travelled together in a non-pickle artifact.
- The reloaded model passed raw/wrapped hazard checks and worked unchanged in both `solve()` and `simulate()`.
- Six-year terminal-state proportions from the validation data were compared directly with case-mix-matched solved and simulated proportions.
- Deployment resolution was checked separately from fitting integration.

For real data, the next additions are exposure-weighted calibration by edge and subgroup, temporal holdouts, cashflow convergence for material benefits, and stress grids based on the actual valuation portfolio.